# ECE1508: Deep Generative Models -- Summer 2026
## Assignment 3: Generative Adversarial Networks
## Question 2: Vanilla GAN on Swiss Roll

In this assignment, we train a vanilla GAN on the 2D Swiss-roll dataset. The goal is to make the alternating min-max loop concrete and to visualize how training behaves before moving to a more realistic dataset.


### Loading Modules
Let's load some required modules.

In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset


# Device
if torch.backends.mps.is_available():
    device = torch.device('mps')
elif torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

print('Device:', device)


### Swiss Roll Dataset

We use a 2D projection of the Swiss-roll manifold. We chose this data, as it is cheap to generate. To synthesize the dataset, we could use `make_swiss_roll` in the `sklearn.datasets` module.


In [ ]:
from sklearn.datasets import make_swiss_roll

# Complete the function that synthesize a dataset with `n_samples` samples
def make_swiss_roll_2d(n_samples=2048, noise=0.15, standardize=True):
    ## COMPLETE ##
    

# Generate a dataset
x_data = make_swiss_roll_2d(n_samples=2048, noise=0.15, standardize=True)


# Plot the scattering diagram of the dataset
plt.figure(figsize=(5, 5))
## COMPLETE ##
plt.title('Swiss-roll data')
plt.axis('equal')
plt.show()


#### Data Loader

Now, let's put the dataset in batches and build the loaders.


In [ ]:
batch_size = 256

tensor_data = ## COMPLETE ##
dataset = ## COMPLETE ##
loader = ## COMPLETE ##



### Models
We now implement simple generator and discriminator to train via min-max game on this dataset. 

#### Discriminator
The discriminator is to get a 2D input $x$ and classify it as `real` or `fake`. We consider the following MLP:
$$
\mathrm{Linear} (2 \rightarrow 64) + \mathrm{LeakyReLU}\\
\downarrow\\
\mathrm{Linear} (64 \rightarrow 64) + \mathrm{LeakyReLU}\\
\downarrow\\
\mathrm{Linear} (64 \rightarrow 64) + \mathrm{LeakyReLU}\\
\downarrow\\
\mathrm{Linear} (64 \rightarrow 1)
$$
For all $\mathrm{LeakyReLU}$ activations set $\texttt{negative\_slope}=0.2$. Note that the last layer should be properly activated for classification.

In [ ]:
class Discriminator(nn.Module):
    def __init__(self):
        ## COMPLETE ##

    def forward(self, x):
        ## COMPLETE ##

#### Generator

For generator, we need a network which converts a 8D noise sample $z$ to a 2D data sample $x$. For this, we implement the following MLP:
$$
\mathrm{Linear} (8 \rightarrow 64) + \mathrm{ReLU}\\
\downarrow\\
\mathrm{Linear} (64 \rightarrow 64) + \mathrm{ReLU}\\
\downarrow\\
\mathrm{Linear} (64 \rightarrow 64) + \mathrm{ReLU}\\
\downarrow\\
\mathrm{Linear} (64 \rightarrow 2) 
$$


In [ ]:
class Generator(nn.Module):
    def __init__(self, z_dim = 8):
        ## COMPLETE ##

    def forward(self, z):
        ## COMPLETE ##



#### Sampling Loop

We finally write a function to sample the generator.


In [ ]:
# Sample a noise sample z from N(0,1)
def sample_noise(n):
    ## COMPLETE ##

# Pass it through the generator and plot the output
def plot_samples(generator, title='Generated samples'):
    generator.eval()
    ## COMPLETE ##


### Training via Min-Max Game

We now implement the training loop. Recall that this is a nested two-tier loop. We consider a general nested loop with `n_inner` inner loop iterations for each outer iteration. 

In [ ]:
def train_GAN(n_inner = 2, z_dim = 8, epochs = 400):

    # Define the loss needed for computing the objective of vanilla GAN
    ## COMPLETE ##

    # instantiate the discriminator
    D = Discriminator().to(device)
    
    # instantiate the generator
    G = Generator(z_dim = z_dim).to(device)

    # Set learning rates
    lr_D = 2e-4
    lr_G = 2e-4

    # Set the optimizers
    opt_G = torch.optim.Adam(G.parameters(), lr=lr_G, betas=(0.5, 0.999))
    opt_D = torch.optim.Adam(D.parameters(), lr=lr_D, betas=(0.5, 0.999))


    history = {
        'd_loss': [],
        'g_loss': [],
        'd_real': [],
        'd_fake': [],
    }

    # start the loop
    for epoch in range(1, epochs + 1):
        ## COMPLETE ##

        for (x_real,) in loader:
            ## COMPLETE ##

            # Discriminator inner loop
            for _ in range (n_inner):
                ## COMPLETE ##
        

            # Generator Update
            ## COMPLETE ##

            with torch.no_grad():
                d_real_prob = ## COMPLETE ##
                d_fake_prob = ## COMPLETE ##

            d_loss_epoch += ## COMPLETE ##
            g_loss_epoch += ## COMPLETE ##
            d_real_epoch += ## COMPLETE ##
            d_fake_epoch += ## COMPLETE ##
            num_batches += ## COMPLETE ##

        # Update history
        ## COMPLETE ##

        if epoch % 100 == 0 or epoch == 1:
            print(f'Epoch {epoch:4d}/{epochs} | D loss: {history["d_loss"][-1]:.4f} | G loss: {history["g_loss"][-1]:.4f} | D(real): {history["d_real"][-1]:.3f} | D(fake): {history["d_fake"][-1]:.3f}')
    return G, history


### Numerical Experiments and Findings

We now train GAN on our dataset and check out the impact of various parameters.


#### Visualizing Generator
Let's first train the model for `n_inner = 2` and sample it to see the learned distribution.

In [ ]:
# train the model
G, history = train_GAN(n_inner = 2)

# use `plot_samples` to sample the trained model and plot it along with the true dataset
## COMPLETE ##



#### Loss Dynamics
We next plot the loss of the loss used by discriminator and generator. Note that when we train the generator, we could ignore the term corresponding to real samples. Also, we should take care of the inner loop's gradient by using the right sign. 

In [ ]:
# Plot loss of G and D against epochs
plt.figure(figsize=(8, 4))
## COMPLETE ##
plt.legend()
plt.show()

## Question: _Explain your observation._
_## COMPLETE ##_

#### Confidence
We can now look at the discriminator's output to understand the confidence of the discriminator after training. 

In [ ]:
# Plot D(real) and D(fake) against epochs
plt.figure(figsize=(8, 4))
## COMPLETE ##
plt.legend()
plt.show()

## Question: _Explain your observation. Does it make sense?_
_## COMPLETE ##_

#### Impact of Inner Loop Convergence

Now we train the model with different `n_inner` and look at the visualization of each trained model.



In [ ]:
# Put it in a loop and look at the visualizations
for n_inner  in [1,2,5,10]:
    ## COMPLETE ##


## Question: _Report your observation and explain it._
_## COMPLETE ##_

#### Impact of Latent Space Dimension
We now play with the latent space dimension to see how it impacts the learning.

In [ ]:
# Put it in a loop and look at the visualizations
for z_dim in [1,2,4,8]:
    ## COMPLETE ##

## Question: _Report your observation and explain it._
_## COMPLETE ##_